# 전처리 1단계 — 스케일 변환 (÷10)

## 왜 이 작업이 필요한가?

> 기상청은 데이터를 저장할 때 파일 크기를 줄이기 위해  
> **소수점을 없애고 10배 곱한 정수**로 저장합니다.  
>  
> 예: 실제 기온 **12.5°C** → 저장값 **125** (÷10 해야 원래 값 복원)  
>  
> EDA 1단계에서 이 사실을 확인했고,  
> 이 작업에서 모든 수치를 **실제 단위로 변환**한 뒤 새 파일로 저장합니다.

## 변환 대상

| 컬럼 | 변환 전 (저장값) | 변환 후 (실제값) | 단위 |
|------|----------------|-----------------|------|
| ta_mean, ta_max | 예: 125 | 12.5 | °C |
| hm_mean, hm_min | 예: 650 | 65.0 | % |
| td_mean, td_min | 예: -28 | -2.8 | °C |
| wind_ws_mean, wind_ws_max | 예: 35 | 3.5 | m/s |
| wind_uu_mean, wind_vv_mean | 예: -17 | -1.7 | m/s |
| rn_day_mean, rn_day_max | 예: 15 | 1.5 | mm |
| **wind_wd_sin_mean, wind_wd_cos_mean** | 그대로 | 그대로 | -1~1 (이미 정상) |

## 결과 저장 위치
```
003. 전처리 데이터/
  └─ weather_scaled/          ← 새로 생성 (월별 파티션)
       ├─ month=2025-02/
       ├─ month=2025-03/
       ├─ month=2025-04/
       └─ month=2025-05/
```

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE_PATH    = r'D:\prj_2_공모전\data\003. 전처리 데이터(preprocessing_data)'
INPUT_PATH   = os.path.join(BASE_PATH, 'grid_date_master')
OUTPUT_PATH  = os.path.join(BASE_PATH, 'weather_scaled')
MONTHS       = ['2025-02', '2025-03', '2025-04', '2025-05']

# ÷10 적용 대상 컬럼
SCALE10_COLS = [
    'ta_mean', 'ta_max',
    'hm_mean', 'hm_min',
    'td_mean', 'td_min',
    'wind_ws_mean', 'wind_ws_max',
    'wind_uu_mean', 'wind_vv_mean',
    'rn_day_mean',  'rn_day_max'
]

# 변환하지 않는 컬럼 (이미 정상 범위)
KEEP_COLS = ['wind_wd_sin_mean', 'wind_wd_cos_mean']

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'출력 경로: {OUTPUT_PATH}')
print('설정 완료')

출력 경로: D:\prj_2_공모전\data\003. 전처리 데이터(preprocessing_data)\weather_scaled
설정 완료


---
## Step 1. 변환 전/후 미리보기

실제로 변환하기 전에, 변환이 올바른지 3행만 확인해봅니다.

In [2]:
df_preview = pd.read_parquet(INPUT_PATH, filters=[('month','==','2025-02')])
df_preview = df_preview.head(3).copy()

print('=== 변환 전 (원본값) ===')
print(df_preview[SCALE10_COLS].to_string())

print('\n=== 변환 후 (÷10 적용) ===')
df_check = df_preview[SCALE10_COLS] / 10
print(df_check.to_string())

print('\n=== 변환 예시 해석 ===')
row = df_preview.iloc[0]
print(f'  기온 (ta_mean) : {row["ta_mean"]:>8.3f}  →  ÷10  →  {row["ta_mean"]/10:>7.3f} °C')
print(f'  습도 (hm_mean) : {row["hm_mean"]:>8.3f}  →  ÷10  →  {row["hm_mean"]/10:>7.3f} %')
print(f'  강수 (rn_day)  : {row["rn_day_mean"]:>8.3f}  →  ÷10  →  {row["rn_day_mean"]/10:>7.3f} mm')
print(f'  풍속 (ws_mean) : {row["wind_ws_mean"]:>8.3f}  →  ÷10  →  {row["wind_ws_mean"]/10:>7.3f} m/s')

=== 변환 전 (원본값) ===
   ta_mean  ta_max  hm_mean  hm_min  td_mean  td_min  wind_ws_mean  wind_ws_max  wind_uu_mean  wind_vv_mean  rn_day_mean  rn_day_max
0   -9.375    59.0  881.750   631.0  -29.375   -43.0         3.750          6.0         -1.75        -0.250        0.375         1.0
1  -10.750    58.0  879.875   632.0  -30.250   -44.0         3.375          6.0         -1.50        -0.375        0.375         1.0
2  -10.750    58.0  879.875   632.0  -30.250   -44.0         3.375          6.0         -1.50        -0.375        0.375         1.0

=== 변환 후 (÷10 적용) ===
   ta_mean  ta_max  hm_mean  hm_min  td_mean  td_min  wind_ws_mean  wind_ws_max  wind_uu_mean  wind_vv_mean  rn_day_mean  rn_day_max
0  -0.9375     5.9  88.1750    63.1  -2.9375    -4.3        0.3750          0.6        -0.175       -0.0250       0.0375         0.1
1  -1.0750     5.8  87.9875    63.2  -3.0250    -4.4        0.3375          0.6        -0.150       -0.0375       0.0375         0.1
2  -1.0750     5.8  87.9875

---
## Step 2. 스케일 변환 실행 및 저장

> 메모리 문제 때문에 **월별로 하나씩** 읽고 → 변환 → 저장합니다.  
> 총 3276만 행이라 한 번에 처리하면 컴퓨터가 멈출 수 있어요.  
>  
> 저장 방식은 원본과 동일하게 **월별 폴더로 나눠서** 저장합니다.

In [3]:
import time
start_total = time.time()

for m in MONTHS:
    start_m = time.time()
    print(f'[{m}] 처리 중...', end=' ')

    # 1. 읽기
    df = pd.read_parquet(INPUT_PATH, filters=[('month', '==', m)])

    # 2. ÷10 스케일 변환
    for col in SCALE10_COLS:
        df[col] = (df[col] / 10).astype('float32')  # float32로 저장 (크기 절약)

    # 3. month 컬럼 제거 (파티션 키라 저장 시 불필요)
    df_save = df.drop(columns=['month'])

    # 4. 월별 폴더에 저장
    out_folder = os.path.join(OUTPUT_PATH, f'month={m}')
    os.makedirs(out_folder, exist_ok=True)
    out_file = os.path.join(out_folder, 'data.parquet')
    df_save.to_parquet(out_file, index=False, engine='pyarrow', compression='snappy')

    elapsed = time.time() - start_m
    file_size = os.path.getsize(out_file) / 1024 / 1024
    print(f'완료 ({len(df):,}행, {file_size:.1f}MB, {elapsed:.1f}초)')

print(f'\n전체 완료: {time.time()-start_total:.1f}초')
print(f'저장 위치: {OUTPUT_PATH}')

[2025-02] 처리 중... 완료 (7,644,028행, 134.0MB, 10.5초)
[2025-03] 처리 중... 완료 (8,463,031행, 150.6MB, 12.3초)
[2025-04] 처리 중... 완료 (8,190,030행, 144.6MB, 12.1초)
[2025-05] 처리 중... 완료 (8,463,031행, 151.0MB, 10.9초)

전체 완료: 45.8초
저장 위치: D:\prj_2_공모전\data\003. 전처리 데이터(preprocessing_data)\weather_scaled


---
## Step 3. 변환 결과 검증

> 저장한 파일을 다시 열어서 변환이 제대로 됐는지 확인합니다.  
> **기온이 -20~30°C 범위**, **습도가 0~100%** 안에 있으면 성공입니다.

In [4]:
# 저장된 파일 불러오기
df_verify = pd.read_parquet(OUTPUT_PATH)

print(f'=== 검증: 저장된 데이터 기본 정보 ===')
print(f'행 수: {len(df_verify):,}')
print(f'컬럼 수: {len(df_verify.columns)}')
print(f'컬럼 목록: {list(df_verify.columns)}')

print()
print('=== 변환 후 범위 검증 (전체 기준) ===')
print(f'{"컬럼":<22} {"최솟값":>9} {"평균":>9} {"최댓값":>9}  판정')
print('-' * 65)

# 실제 단위 기준 허용 범위
expected = {
    'ta_mean':     (-30, 45,  '°C'),
    'ta_max':      (-30, 45,  '°C'),
    'hm_mean':     (0,   100, '%'),
    'hm_min':      (0,   100, '%'),
    'td_mean':     (-40, 35,  '°C'),
    'td_min':      (-40, 35,  '°C'),
    'wind_ws_mean':(0,   50,  'm/s'),
    'wind_ws_max': (0,   50,  'm/s'),
    'wind_uu_mean':(-50, 50,  'm/s'),
    'wind_vv_mean':(-50, 50,  'm/s'),
    'rn_day_mean': (0,   200, 'mm'),
    'rn_day_max':  (0,   500, 'mm'),
}

all_ok = True
for col, (lo, hi, unit) in expected.items():
    v_min = df_verify[col].min()
    v_mean = df_verify[col].mean()
    v_max = df_verify[col].max()
    ok = (v_min >= lo) and (v_max <= hi)
    flag = 'OK' if ok else 'NG!!'
    if not ok:
        all_ok = False
    print(f'{col:<22} {v_min:>9.2f} {v_mean:>9.2f} {v_max:>9.2f}  [{flag}] ({unit})')

print()
if all_ok:
    print('=> 모든 컬럼 범위 정상 -- 스케일 변환 성공!')
else:
    print('=> 일부 컬럼 범위 이상 -- 확인 필요')

=== 검증: 저장된 데이터 기본 정보 ===
행 수: 32,760,120
컬럼 수: 19
컬럼 목록: ['grid_id', 'kma_nx', 'kma_ny', 'ta_mean', 'ta_max', 'hm_mean', 'hm_min', 'td_mean', 'td_min', 'wind_ws_mean', 'wind_ws_max', 'wind_uu_mean', 'wind_vv_mean', 'wind_wd_sin_mean', 'wind_wd_cos_mean', 'rn_day_mean', 'rn_day_max', 'date', 'month']

=== 변환 후 범위 검증 (전체 기준) ===
컬럼                           최솟값        평균       최댓값  판정
-----------------------------------------------------------------
ta_mean                   -19.36      6.94     27.16  [OK] (°C)
ta_max                    -16.20     12.32     32.10  [OK] (°C)
hm_mean                    14.07     62.13    100.00  [OK] (%)
hm_min                      4.30     40.47    100.00  [OK] (%)
td_mean                   -25.77     -0.97     20.76  [OK] (°C)
td_min                    -34.20     -4.01     20.40  [OK] (°C)
wind_ws_mean                0.05      1.33      8.93  [OK] (m/s)
wind_ws_max                 0.10      2.45     23.50  [OK] (m/s)
wind_uu_mean               -5.88   

In [5]:
# 샘플 3행 출력
print('=== 변환 후 샘플 (상위 3행) ===')
df_verify.head(3)

=== 변환 후 샘플 (상위 3행) ===


,grid_id,kma_nx,kma_ny,ta_mean,ta_max,hm_mean,hm_min,td_mean,td_min,wind_ws_mean,wind_ws_max,wind_uu_mean,wind_vv_mean,wind_wd_sin_mean,wind_wd_cos_mean,rn_day_mean,rn_day_max,date,month
0,10007_19696,1139,1483,-0.9375,5.9,88.175003,63.099998,-2.9375,-4.3,0.3750,0.6,-0.175,-0.0250,0.484131,0.226221,0.0375,0.1,2025-02-01,2025-02
1,10008_19691,1139,1482,-1.0750,5.8,87.987503,63.200001,-3.0250,-4.4,0.3375,0.6,-0.150,-0.0375,0.252132,0.023642,0.0375,0.1,2025-02-01,2025-02
2,10008_19692,1139,1482,-1.0750,5.8,87.987503,63.200001,-3.0250,-4.4,0.3375,0.6,-0.150,-0.0375,0.252132,0.023642,0.0375,0.1,2025-02-01,2025-02


---
## Step 4. 파일 크기 비교

> float32(32비트)로 저장하면 float64(64비트)보다 파일 크기가 절반 정도 작아집니다.  
> 날씨 데이터 정밀도(소수점 1~2자리)에는 float32로 충분합니다.

In [6]:
def get_folder_size_mb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024 / 1024

orig_size  = get_folder_size_mb(INPUT_PATH)
new_size   = get_folder_size_mb(OUTPUT_PATH)

print(f'원본 (grid_date_master): {orig_size:.1f} MB')
print(f'변환 (weather_scaled)  : {new_size:.1f} MB')
print(f'크기 변화               : {new_size/orig_size*100:.1f}%  ({"감소" if new_size < orig_size else "증가"})')

원본 (grid_date_master): 595.2 MB
변환 (weather_scaled)  : 580.2 MB
크기 변화               : 97.5%  (감소)


---
## 최종 요약

In [7]:
print('=' * 55)
print('  전처리 1단계 — 스케일 변환 완료')
print('=' * 55)
print(f'''
  변환 내용
  - 12개 수치 컬럼에 ÷10 적용
  - wind_wd_sin/cos 2개는 그대로 유지
  - float64 -> float32로 저장 (크기 절약)

  결과 파일
  - 위치: weather_scaled/ (월별 파티션)
  - 읽기: pd.read_parquet("weather_scaled")

  다음 단계
  - 파생변수 생성
    (1) 실효습도: 현재+과거 4일 평균습도 가중합
    (2) 3일 누적 강수량: RNE_Temp (DWI 계산용)
    (3) 일 가중치(w_d): 월/일 기준 산불 위험 계수
''')
print('=' * 55)

  전처리 1단계 — 스케일 변환 완료

  변환 내용
  - 12개 수치 컬럼에 ÷10 적용
  - wind_wd_sin/cos 2개는 그대로 유지
  - float64 -> float32로 저장 (크기 절약)

  결과 파일
  - 위치: weather_scaled/ (월별 파티션)
  - 읽기: pd.read_parquet("weather_scaled")

  다음 단계
  - 파생변수 생성
    (1) 실효습도: 현재+과거 4일 평균습도 가중합
    (2) 3일 누적 강수량: RNE_Temp (DWI 계산용)
    (3) 일 가중치(w_d): 월/일 기준 산불 위험 계수

